In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import metpy
from matplotlib import ticker

## Vertical distribution

In [ ]:
ds = xr.open_mfdataset('/Path/to/data')
#print(list(ds.variables))
tr50  = ds.tracer50_chemtr
tr500 = ds.tracer500_chemtr
lat = ds.lat
lon = ds.lon
time = np.arange(0,25) #in UTC
z = ds.z_mc

In [ ]:
tr50_karlsruhe = tr50.sel(lon=8.4,lat=49.0,method='nearest')*1e9
tr500_karlsruhe= tr500.sel(lon=8.4,lat=49.0,method='nearest')*1e9
z_karlsruhe = z.sel(lon=8.4,lat=49.0,method='nearest')

fig1 = plt.figure(figsize=(12,8))
fig1.subplots_adjust(wspace=0.3,hspace=0.3)
ax1 = plt.subplot(2,2,1)
log_lev = np.array([1e0,5e0,1e1,5e1,1e2,5e2,1e3,5e3,1e4])
plt.contourf(time,z_karlsruhe[0,:]/1000,tr50_karlsruhe.transpose(),cmap='Blues',locator=ticker.LogLocator(),levels=log_lev,extend='both')
plt.ylim([0.15,2])
plt.colorbar()
plt.ylabel('height in km')
plt.vlines(4.92,0,2,color='k',linestyle='dotted')
plt.vlines(18.85,0,2,color='k',linestyle='dotted')
plt.title('Concentration above Karlsruhe in kg/kg')
plt.xlabel('Time in UTC')

ax1 = plt.subplot(2,2,2)
plt.plot(time,tr50_karlsruhe[:,-1])
plt.xlim([1,24])
plt.title('50 m Emission: Surface concentration in kg/kg')
plt.ylabel('Conc. in kg/kg')
plt.xlabel('Time in UTC')

ax1 = plt.subplot(2,2,3)
plt.contourf(time,z_karlsruhe[0,:]/1000,tr500_karlsruhe.transpose(),cmap='Blues',locator=ticker.LogLocator(),levels=log_lev,extend='both')
#levels=np.arange(0,1000,50),extend='max'
plt.ylim([0.15,2])
plt.colorbar()
plt.ylabel('height in km')
plt.vlines(4.92,0,2,color='k',linestyle='dotted')
plt.vlines(18.85,0,2,color='k',linestyle='dotted')
plt.xlabel('Time in UTC')
plt.title('Concentration above Karlsruhe in kg/kg')

ax1 = plt.subplot(2,2,4)
plt.plot(time,tr500_karlsruhe[:,-1])
plt.xlim([1,24])
plt.title('500 m Emission: Surface concentration in kg/kg')
plt.xlabel('Time in UTC')
plt.ylabel('Conc. in kg/kg')

## Horizontal distribution

In [ ]:
def get_dz(ds):
    dz = -1 * ds.z_ifc.diff('height_3')
    dz = dz.rename({'height_3':'height'})
    dz = dz.assign_coords(height=(dz.height - 1))
    return dz

tr50_load = (tr50 * ds.rho * get_dz(ds)).sum('height')*1e3
tr500_load= (tr500 * ds.rho * get_dz(ds)).sum('height')*1e3
tr50_load = tr50_load.where(tr50_load>0)
tr500_load = tr500_load.where(tr500_load>0)

In [ ]:
crs_proj = ccrs.PlateCarree()
log_lev = np.array([1e-3,5e-3,1e-2,5e-2,1e-1,5e-1,1e0,5e0,1e1])

fig1 = plt.figure(figsize=(9,6))
ax1 = plt.subplot(2,2,1,projection=crs_proj)
c=plt.contourf(ds.lon,ds.lat,tr50_load[12,:,:],cmap='Blues',transform=crs_proj,locator=ticker.LogLocator(),levels=log_lev,extend='both')
ax1.coastlines(resolution='50m')
plt.title('Emission at 50 m after 12 h')

ax1 = plt.subplot(2,2,2,projection=crs_proj)
c=plt.contourf(ds.lon,ds.lat,tr500_load[12,:,:],cmap='Blues',transform=crs_proj,locator=ticker.LogLocator(),levels=log_lev,extend='both')
ax1.coastlines(resolution='50m')
plt.title('Emission at 500 m after 12 h')

ax1 = plt.subplot(2,2,3,projection=crs_proj)
c=plt.contourf(ds.lon,ds.lat,tr50_load[24,:,:],cmap='Blues',transform=crs_proj,locator=ticker.LogLocator(),levels=log_lev,extend='both')
ax1.coastlines(resolution='50m')
plt.title('Emission at 50 m after 24 h')

ax1 = plt.subplot(2,2,4,projection=crs_proj)
c=plt.contourf(ds.lon,ds.lat,tr500_load[24,:,:],cmap='Blues',transform=crs_proj,locator=ticker.LogLocator(),levels=log_lev,extend='both')
ax1.coastlines(resolution='50m')
plt.title('Emission at 500 m after 24 h')

ax2=fig1.add_axes([0.2,0.05,0.6,0.03])
cbar=plt.colorbar(c,cax=ax2,orientation='horizontal')
cbar.set_label('tracer column loading in g/m²')

## Richardson number 

In [ ]:
theta_karlsruhe = (ds.theta_v).sel(lon=8.4,lat=49.0,method='nearest')
u = (ds.u).sel(lon=8.4,lat=49.0,method='nearest')
v = (ds.v).sel(lon=8.4,lat=49.0,method='nearest')
ri=metpy.calc.gradient_richardson_number(z_karlsruhe[:,:], theta_karlsruhe[:,:], u[:,:], v[:,:], vertical_dim=1)

In [ ]:
plt.plot(ri[6,:],z_karlsruhe[0,:]/1000,label='after 6 h')
plt.plot(ri[12,:],z_karlsruhe[0,:]/1000,label='after 12 h')
plt.plot(ri[18,:],z_karlsruhe[0,:]/1000,label='after 18 h')
plt.plot(ri[24,:],z_karlsruhe[0,:]/1000,label='after 24 h')
plt.vlines(0.25,0,2,color='k',linestyle='dashed')
plt.ylim([0.15,2])
plt.xlim([-4,10])
plt.legend()
plt.ylabel('height in km')

## Lapse rate

In [ ]:
fig1 = plt.figure(figsize=(12,8))
fig1.subplots_adjust(wspace=0.3,hspace=0.3)
ax1 = plt.subplot(2,2,1)
t_karlsruhe = (ds.temp).sel(lon=8.4,lat=49.0,method='nearest')
plt.plot(t_karlsruhe[6,:],z_karlsruhe[0,:]/1000,label='after 6 h')
plt.plot(t_karlsruhe[12,:],z_karlsruhe[0,:]/1000,label='after 12 h')
plt.plot(t_karlsruhe[18,:],z_karlsruhe[0,:]/1000,label='after 18 h')
plt.plot(t_karlsruhe[24,:],z_karlsruhe[0,:]/1000,label='after 24 h')
plt.ylim([0.15,2])
plt.xlim([280,298])
plt.ylabel('height in km')
plt.xlabel('Temperature in K')

ax1 = plt.subplot(2,2,2)
plt.contourf(time,z_karlsruhe[0,:]/1000,t_karlsruhe.transpose(),levels=np.arange(280,298,1),extend='both')
plt.ylim([0.15,2])
plt.colorbar()
plt.title('Temperature in K')
plt.ylabel('Height in km')
plt.xlabel('Time in UTC')

ax1 = plt.subplot(2,2,3)
theta_karlsruhe = (ds.theta_v).sel(lon=8.4,lat=49.0,method='nearest')
plt.plot(theta_karlsruhe[6,:],z_karlsruhe[0,:]/1000,label='after 6 h')
plt.plot(theta_karlsruhe[12,:],z_karlsruhe[0,:]/1000,label='after 12 h')
plt.plot(theta_karlsruhe[18,:],z_karlsruhe[0,:]/1000,label='after 18 h')
plt.plot(theta_karlsruhe[24,:],z_karlsruhe[0,:]/1000,label='after 24 h')
plt.legend()
plt.ylim([0.15,2])
plt.xlim([285,300])
plt.ylabel('height in km')
plt.xlabel('Virtual Potential Temperature in K')

ax1 = plt.subplot(2,2,4)
plt.contourf(time,z_karlsruhe[0,:]/1000,theta_karlsruhe.transpose(),levels=np.arange(285,300,1),extend='both')
plt.ylim([0.15,2])
plt.colorbar()
plt.title('Virtual Potential Temperature in K')
plt.ylabel('Height in km')
plt.xlabel('Time in UTC')

## Wind speed

In [ ]:
fig1 = plt.figure(figsize=(9,9))
ax1 = plt.subplot(3,1,1)

# horizontal wind speed
wind_speed = np.sqrt(ds.u**2 + ds.v**2)
wind_speed = wind_speed.sel(lon=8.4,lat=49.0,method='nearest')
plt.contourf(time,z_karlsruhe[0,:]/1000,wind_speed.transpose(),cmap='Reds',levels=np.arange(0,15,1),extend='max')
plt.colorbar()
plt.ylim([0.15,2])
plt.ylabel('height in km')
plt.vlines(4.92,0,2,color='k',linestyle='dotted')
plt.vlines(18.85,0,2,color='k',linestyle='dotted')

# vertical wind speed
ax1 = plt.subplot(3,1,2)
w = (ds.w).sel(lon=8.4,lat=49.0,method='nearest')
zifc_karlsruhe = (ds.z_ifc).sel(lon=8.4,lat=49.0,method='nearest')
plt.contourf(time,zifc_karlsruhe[0,:]/1000,w.transpose(),cmap='PiYG',levels=np.arange(-0.1,0.11,0.02),extend='both')
plt.colorbar()
plt.ylim([0.15,2])
plt.ylabel('height in km')
plt.vlines(4.92,0,2,color='k',linestyle='dotted')
plt.vlines(18.85,0,2,color='k',linestyle='dotted')

# TKE
tke = ds.tke
ax1 = plt.subplot(3,1,3)
tke = tke.sel(lon=8.4,lat=49.0,method='nearest')
plt.contourf(time,zifc_karlsruhe[0,:]/1000,tke.transpose(),cmap='Greens',levels=np.arange(0,3,0.2),extend='max')
plt.colorbar()
plt.ylim([0.15,2])
plt.xlabel('Time in UTC')
plt.ylabel('height in km')
plt.vlines(4.92,0,2,color='k',linestyle='dotted')
plt.vlines(18.85,0,2,color='k',linestyle='dotted')
